# 00｜构建 ViT Universe

Universe 以 T 日状态定义，只剔除停牌、涨停、跌停和状态缺失股票；不处理 ST 或上市时间。T+1 状态单独保存为执行掩码。

## 步骤 1：定位统一流水线并设置参数

In [ ]:
from pathlib import Path
import os
import sys

CURRENT_DIR = Path.cwd().resolve()
VIT_DIR = CURRENT_DIR if (CURRENT_DIR / 'vit_stage02_universe.py').exists() else CURRENT_DIR / 'ViT'
if not (VIT_DIR / 'vit_stage02_universe.py').exists():
    raise FileNotFoundError(f'找不到 Universe 模块：{VIT_DIR / "vit_stage02_universe.py"}')
if str(VIT_DIR) not in sys.path:
    sys.path.insert(0, str(VIT_DIR))
from vit_stage02_universe import build_universe

# 0 表示签名一致时复用；正式运行不限制日期。
os.environ['VIT_UNIVERSE_OVERWRITE'] = '0'
os.environ.pop('VIT_MAX_IMAGE_DATES', None)
print({'ViT目录': str(VIT_DIR), '覆盖Universe': False, '日期上限': None})

## 步骤 2：构建年度 Universe

In [ ]:
universe_result = build_universe()
print(universe_result)

## 步骤 3：查看每日质量与年度汇总

In [ ]:
import json
import pandas as pd
from IPython.display import display

manifest = json.loads((VIT_DIR / 'data/universe/universe_manifest.json').read_text(encoding='utf-8'))
quality = pd.read_parquet(VIT_DIR / 'data/quality/universe_quality_by_date.parquet')
quality['year'] = quality['trade_date'].astype(str).str[:4]
display(pd.Series(manifest, name='value').to_frame())
display(quality.groupby('year', as_index=False).sum(numeric_only=True))